# Sable-7B: Fast Fine-Tuning Qwen2.5-Coder-7B on Kaggle (Free T4 GPU)

**Hardware**: Kaggle NVIDIA Tesla T4
**Framework**: Unsloth / Hugging Face TRL + PEFT (4-bit QLoRA)
**Dataset**: 25,000 curated high-quality instruction pairs (Optimal for SFT)
**Estimated Training Time**: ~30 to 45 minutes total

---
### Settings in Right Sidebar:
1. **Accelerator**: `GPU T4 x 2`
2. **Internet**: `On`
3. **Input**: Attach `sable-dataset`

In [ ]:
# 1. Install Unsloth and dependencies
!pip install --no-deps bitsandbytes accelerate peft trl triton
!pip install unsloth

In [ ]:
# 2. Load Qwen2.5-Coder-7B-Instruct with 4-bit Quantization
import os, torch
# Place entire model on GPU 0 to eliminate cross-GPU PCIe communication latency
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel

max_seq_length = 1024
dtype = None # Auto-detects float16 for Tesla T4
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0},
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Cuts VRAM significantly
    random_state = 42,
)

# Enable faster training mode
FastLanguageModel.for_training(model)

In [ ]:
# 4. Load Dataset from Kaggle Input
import glob, os
from datasets import load_dataset

# Locate train.jsonl in Kaggle input directory (handles zip or raw files)
train_files = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
if not train_files:
    raise FileNotFoundError("Please attach your 'sable-dataset' in the Kaggle right sidebar (+ Add Input)!")

train_path = train_files[0]
print(f"Found dataset at: {train_path}")

dataset = load_dataset("json", data_files={"train": train_path})
print(f"Total raw samples in dataset: {len(dataset['train'])}")

# Sample 25,000 high-quality records (Standard for SFT fine-tuning)
MAX_SAMPLES = 25000
if len(dataset["train"]) > MAX_SAMPLES:
    dataset["train"] = dataset["train"].shuffle(seed=42).select(range(MAX_SAMPLES))
    print(f"Curated {MAX_SAMPLES} samples for fast optimal fine-tuning (~35 mins runtime)")
else:
    print(f"Using all {len(dataset['train'])} samples")

# Format chat messages into text using Qwen's native chat template
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)

In [ ]:
# 5. Configure Training Arguments
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8, # Effective batch size = 16
        warmup_ratio = 0.03,
        num_train_epochs = 1, # 1 epoch on 25k samples = 1,562 steps (~35 mins)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "/kaggle/working/sable_output",
        save_steps = 250,
        save_total_limit = 2,
    ),
)

# 6. Start Training
print("Starting Fast Single-GPU Training on Kaggle...")
trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
# 7. Save Final LoRA Adapters & Create Download ZIP
import shutil
output_dir = "/kaggle/working/sable_lora_final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Training successfully finished! Adapters saved to {output_dir}")

# Create single ZIP archive for 1-click download
shutil.make_archive("/kaggle/working/sable_lora_final", 'zip', output_dir)
print("Created archive: /kaggle/working/sable_lora_final.zip (Ready for download!)")

In [ ]:
# 8. Test Your Fine-Tuned Model Immediately!
FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "Write a Python function to solve the two-sum problem in O(n) time, with docstring and type hints."}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
print("--- Model Output ---")
print(tokenizer.batch_decode(outputs)[0])